In [9]:
# -*- coding: utf-8 -*-
"""
SKRIP FINAL PENELITIAN: RANDOM FOREST 3 KELAS
Skenario: Scaler vs No Scaler
Visualisasi: Metrik, Regresi, Confusion Matrix, dan SHAP per Kelas
"""

import pandas as pd
import numpy as np
import shap
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, matthews_corrcoef,
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
)

warnings.filterwarnings('ignore')

# ============================================================
# 1. BACA & PREPROCESSING DATA
# ============================================================
print("Memulai proses preprocessing data...")
df = pd.read_csv('data_advanced.csv')

# Drop kolom pemicu Data Leakage & Identitas
cols_to_drop = ['nisn', 'nama_siswa', 'email', 'gender', 'nilai_raport_zscore_jurusan']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
target_col = 'nilai_rata_rata_raport'

for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

# Feature Engineering
df['rasio_belajar_vs_layar'] = df['jam_belajar_per_hari'] / (df['screen_time'] + 1)
df['indeks_produktivitas'] = (df['skor_time_management'] * df['presentase_kehadiran']) / 100
df['sisa_waktu_aktif'] = 24 - (df['jam_tidur'] + df['jam_belajar_per_hari'] + df['screen_time'])

# Numerik
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

for col in numeric_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)

# Ordinal & Nominal
ordinal_mappings = {
    'study_environment': ['Kurang Kondusif', 'Cukup Kondusif', 'Kondusif'],
    'kompetensi_skill_level': ['Rendah', 'Menengah', 'Tinggi'],
    'stress_level': ['Rendah', 'Sedang', 'Berat']
}
nominal_cols = ['kerja_sampingan', 'industry_readiness', 'jurusan']

for col, cats in ordinal_mappings.items():
    if col in df.columns:
        oe = OrdinalEncoder(categories=[cats], dtype=int)
        df[col] = oe.fit_transform(df[[col]])

nominal_cols_exist = [c for c in nominal_cols if c in df.columns]
if nominal_cols_exist:
    onehot_encoder = OneHotEncoder(sparse_output=False, drop=None)
    onehot_encoded = onehot_encoder.fit_transform(df[nominal_cols_exist])
    onehot_feature_names = onehot_encoder.get_feature_names_out(nominal_cols_exist)
    df_onehot = pd.DataFrame(onehot_encoded, columns=onehot_feature_names, index=df.index)
else:
    df_onehot = pd.DataFrame(index=df.index)

# Polynomial Interactions
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_numeric = poly.fit_transform(df[numeric_cols])
poly_feature_names = poly.get_feature_names_out(numeric_cols)

df_poly = pd.DataFrame(poly_numeric, columns=poly_feature_names, index=df.index)
df_ordinal = df[[c for c in ordinal_mappings.keys() if c in df.columns]].copy()

X = pd.concat([df_poly, df_ordinal, df_onehot], axis=1)
feature_names = X.columns.tolist()
y_reg = df[target_col].values

Memulai proses preprocessing data...


In [10]:
# ============================================================
# 2. TARGET ENGINEERING (3 CLASS TERTILES)
# ============================================================
low_3 = df[target_col].quantile(0.33)
high_3 = df[target_col].quantile(0.67)

def get_3class(score):
    if score <= low_3: return 'Sangat Beresiko'
    elif score <= high_3: return 'Beresiko'
    else: return 'Aman'

y_3class = np.array([get_3class(s) for s in df[target_col]])

In [11]:
# ============================================================
# 3. FUNGSI PELATIHAN & EVALUASI
# ============================================================
def run_scenario(X, y_cls, y_reg, use_scaler, scenario_name):
    print(f"\nMenjalankan: {scenario_name} ...")
    
    # Split Data
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, test_idx = next(sss.split(X, y_cls))
    X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
    y_train_cls, y_test_cls = y_cls[train_idx], y_cls[test_idx]
    y_train_reg, y_test_reg = y_reg[train_idx], y_reg[test_idx]

    # Scaling
    if use_scaler:
        scaler = StandardScaler()
        X_train_proc = scaler.fit_transform(X_train)
        X_test_proc = scaler.transform(X_test)
    else:
        scaler = None
        X_train_proc = X_train.values
        X_test_proc = X_test.values

    # Feature Selection (12 Fitur Terbaik)
    selector = SelectKBest(mutual_info_classif, k=12)
    X_train_sel = selector.fit_transform(X_train_proc, y_train_cls)
    X_test_sel = selector.transform(X_test_proc)
    selected_features = np.array(feature_names)[selector.get_support()]

    # KLASIFIKASI (Random Forest)
    class_weight_options = [
        {'Sangat Beresiko': 1.0, 'Beresiko': 2.5, 'Aman': 1.5},
        {'Sangat Beresiko': 1.0, 'Beresiko': 3.0, 'Aman': 1.0}
    ]

    rf_cls = RandomForestClassifier(random_state=42, criterion='entropy', n_jobs=-1)
    param_grid_cls = {
        'class_weight': class_weight_options,
        'n_estimators': [200, 300],
        'max_depth': [5, 7], 
        'min_samples_leaf': [1, 2]
    }
    grid_cls = GridSearchCV(rf_cls, param_grid_cls, cv=5, scoring='f1_macro', n_jobs=-1)
    grid_cls.fit(X_train_sel, y_train_cls)
    best_cls = grid_cls.best_estimator_
    y_pred_cls = best_cls.predict(X_test_sel)

    # Metrics Klasifikasi
    acc = accuracy_score(y_test_cls, y_pred_cls)
    f1_mac = f1_score(y_test_cls, y_pred_cls, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_test_cls, y_pred_cls)
    report = classification_report(y_test_cls, y_pred_cls, output_dict=True, zero_division=0)
    
    recall_str = []
    for c in ['Sangat Beresiko', 'Beresiko', 'Aman']:
        if c in report: recall_str.append(f"{c[:2]}:{report[c]['recall']:.2f}")
    recall_summary = " | ".join(recall_str)

    # REGRESI (Random Forest Regressor)
    rf_reg = RandomForestRegressor(random_state=42, n_jobs=-1)
    param_grid_reg = {'n_estimators': [200], 'max_depth': [5, 7], 'min_samples_leaf': [2]}
    grid_reg = GridSearchCV(rf_reg, param_grid_reg, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    grid_reg.fit(X_train_sel, y_train_reg)
    y_pred_reg = grid_reg.best_estimator_.predict(X_test_sel)

    # Metrics Regresi
    mae = mean_absolute_error(y_test_reg, y_pred_reg)
    r2 = r2_score(y_test_reg, y_pred_reg)

    # SHAP ANALYSIS - Top 3 untuk Tabel
    explainer = shap.TreeExplainer(best_cls)
    shap_values = explainer.shap_values(X_test_sel)
    
    target_class = 'Sangat Beresiko'
    classes = list(best_cls.classes_)
    target_idx = classes.index(target_class) if target_class in classes else 0

    if isinstance(shap_values, list):
        shap_target = shap_values[target_idx]
    else:
        shap_target = shap_values[:, :, target_idx] if len(shap_values.shape) == 3 else shap_values

    shap_importance = np.abs(shap_target).mean(axis=0)
    imp_df = pd.DataFrame({'feature': selected_features, 'importance': shap_importance}).sort_values('importance', ascending=False)
    top_3_features = ", ".join(imp_df.head(3)['feature'].tolist())

    return {
        'Skenario': scenario_name,
        'Accuracy': acc,
        'F1-Macro': f1_mac,
        'MCC': mcc,
        'Recall': recall_summary,
        'MAE': mae,
        'R2': r2,
        'Top 3 SHAP': top_3_features,
        'model': best_cls,
        'scaler': scaler,
        'selector': selector,
        'selected_features': selected_features
    }

In [12]:
# ============================================================
# 4. EKSEKUSI SKENARIO 3 KELAS
# ============================================================
scenarios = [
    (True, "1. Scaler + 3 Class"),
    (False, "2. No Scaler + 3 Class")
]

results = []
for use_sc, name in scenarios:
    res = run_scenario(X, y_3class, y_reg, use_scaler=use_sc, scenario_name=name)
    results.append(res)


Menjalankan: 1. Scaler + 3 Class ...

Menjalankan: 2. No Scaler + 3 Class ...


In [13]:
# ============================================================
# 5. CETAK TABEL PERBANDINGAN
# ============================================================
summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ['model', 'scaler', 'selector', 'selected_features']} 
    for r in results
])

print("\n" + "="*95)
print("=== HASIL AKHIR PENELITIAN: 3 KELAS (SCALER VS NO SCALER) ===")
print("="*95)

best_result = max(results, key=lambda x: x['F1-Macro'])
print(f"\n>> Model Terbaik Terpilih: {best_result['Skenario']} (F1-Macro: {best_result['F1-Macro']:.4f})\n")

summary_df['Accuracy'] = summary_df['Accuracy'].apply(lambda x: f"{x:.4f}")
summary_df['F1-Macro'] = summary_df['F1-Macro'].apply(lambda x: f"{x:.4f}")
summary_df['MCC'] = summary_df['MCC'].apply(lambda x: f"{x:.4f}")
summary_df['MAE'] = summary_df['MAE'].apply(lambda x: f"{x:.4f}")
summary_df['R2'] = summary_df['R2'].apply(lambda x: f"{x:.4f}")

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)
print(summary_df.to_string(index=False))
print("="*95)


=== HASIL AKHIR PENELITIAN: 3 KELAS (SCALER VS NO SCALER) ===

>> Model Terbaik Terpilih: 1. Scaler + 3 Class (F1-Macro: 0.3989)

              Skenario Accuracy F1-Macro     MCC                      Recall    MAE     R2                                                                                                                 Top 3 SHAP
   1. Scaler + 3 Class   0.4000   0.3989  0.0978 Sa:0.50 | Be:0.36 | Am:0.33 1.4141 0.0993 jam_belajar_per_hari indeks_produktivitas, presentase_kehadiran rasio_belajar_vs_layar, screen_time rasio_belajar_vs_layar
2. No Scaler + 3 Class   0.3000   0.2978 -0.0573 Sa:0.43 | Be:0.29 | Am:0.17 1.3339 0.1540              jam_tidur presentase_kehadiran, jam_belajar_per_hari indeks_produktivitas, screen_time rasio_belajar_vs_layar


In [14]:
# ============================================================
# 6. VISUALISASI GRAFIK EVALUASI & SHAP
# ============================================================
print("\nMembuat visualisasi grafik...")
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

nama_skenario = [res['Skenario'] for res in results]
skor_acc = [res['Accuracy'] for res in results]
skor_f1 = [res['F1-Macro'] for res in results]
skor_mcc = [res['MCC'] for res in results]
skor_mae = [res['MAE'] for res in results]
skor_r2 = [res['R2'] for res in results]

# --- FIGURE 1: Klasifikasi ---
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(nama_skenario))
width = 0.25

rects1 = ax.bar(x - width, skor_acc, width, label='Accuracy', color='#4C72B0')
rects2 = ax.bar(x, skor_f1, width, label='F1-Macro', color='#55A868')
rects3 = ax.bar(x + width, skor_mcc, width, label='MCC', color='#C44E52')

ax.set_ylabel('Skor', fontsize=12, fontweight='bold')
ax.set_title('Perbandingan Metrik Klasifikasi (3 Kelas)', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(nama_skenario, fontsize=11, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1, 1.1), ncol=3)

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.savefig('grafik_klasifikasi_3class.png', dpi=300)
plt.close()

# --- FIGURE 2: SHAP PER KELAS DARI MODEL TERBAIK ---
print("\nMengekstrak visualisasi SHAP per kelas dari model terbaik...")
sns.set_theme(style="white") 

model = best_result['model']
scaler = best_result['scaler']
selector = best_result['selector']
selected_features = best_result['selected_features']

# Re-create testing set
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(sss.split(X, y_3class))
X_test_raw = X.iloc[test_idx].reset_index(drop=True)

if scaler:
    X_test_proc = scaler.transform(X_test_raw)
else:
    X_test_proc = X_test_raw.values
    
X_test_sel = selector.transform(X_test_proc)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sel)
classes = list(model.classes_)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(f'Pendorong Keputusan (SHAP) per Kelas\n{best_result["Skenario"]}', 
             fontsize=16, fontweight='bold', y=1.05)

for idx, cls_name in enumerate(classes):
    ax = axes[idx]
    
    if isinstance(shap_values, list):
        shap_target = shap_values[idx]
    elif len(shap_values.shape) == 3:
        shap_target = shap_values[:, :, idx]
    else:
        shap_target = shap_values

    shap_importance = np.abs(shap_target).mean(axis=0)
    imp_df = pd.DataFrame({
        'Feature': selected_features, 
        'Importance': shap_importance
    }).sort_values('Importance', ascending=True).tail(8) 
    
    ax.barh(imp_df['Feature'], imp_df['Importance'], color='#C44E52', edgecolor='black')
    ax.set_title(f"Target: {cls_name.upper()}", fontsize=13, fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
    ax.xaxis.grid(True, linestyle='--', alpha=0.7)
    
plt.tight_layout()
plt.savefig('grafik_shap_per_kelas_3class.png', dpi=300, bbox_inches='tight')
plt.close()

print("\nSeluruh tahapan pemrosesan dan visualisasi telah selesai dengan sukses.")


Membuat visualisasi grafik...

Mengekstrak visualisasi SHAP per kelas dari model terbaik...

Seluruh tahapan pemrosesan dan visualisasi telah selesai dengan sukses.
